[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Feature Importance &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


**1.** Compute permutation importance for `forest` with `n_repeats=30` and `random_state=1`, and print the mean drop for each feature rounded to three places, sorted from largest to smallest.

In [2]:
X = network[FEATURES]
y = network["jan_mean"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
forest = RandomForestRegressor(random_state=0).fit(X_train, y_train)

result = permutation_importance(forest, X_test, y_test, n_repeats=30, random_state=1)

print(pd.Series(result.importances_mean, index=FEATURES).sort_values(ascending=False).round(3))

latitude       0.682
elevation_m    0.330
coast_km       0.298
longitude      0.031
installed     -0.002
dtype: float64


The first four lines fit `forest` as the worked example did, since this notebook runs on its own. More repeats and another seed move each mean a little, and the order stays the same.

**2.** Fit `RandomForestRegressor(max_depth=3, random_state=0)` on the training rows, and print its `feature_importances_` beside the deeper forest's, rounded to three places.

In [3]:
X = network[FEATURES]
y = network["jan_mean"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
forest = RandomForestRegressor(random_state=0).fit(X_train, y_train)

shallow = RandomForestRegressor(max_depth=3, random_state=0).fit(X_train, y_train)

print(pd.DataFrame({"depth 3": shallow.feature_importances_, "full depth": forest.feature_importances_},
                   index=FEATURES).round(3))

             depth 3  full depth
latitude       0.561       0.479
longitude      0.033       0.059
elevation_m    0.162       0.196
coast_km       0.243       0.248
installed      0.000       0.018


Trees three levels deep have room for a few splits each, and spend them on the strongest features, so `installed` gets nothing and longitude 0.033 against 0.059.

**3.** Compute permutation importance for `forest` with `scoring="neg_mean_absolute_error"`, and print the mean drops. Each is now how many degrees the mean absolute error grows when the feature is shuffled.

In [4]:
X = network[FEATURES]
y = network["jan_mean"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
forest = RandomForestRegressor(random_state=0).fit(X_train, y_train)

result = permutation_importance(forest, X_test, y_test, n_repeats=10, random_state=0,
                                scoring="neg_mean_absolute_error")

print(pd.Series(result.importances_mean, index=FEATURES).round(3))

latitude       1.342
longitude      0.091
elevation_m    0.724
coast_km       0.792
installed     -0.008
dtype: float64


The scorer is the negative of the error, so a drop in it is a rise in the error, in degrees. The **Regression Metrics** notebook covers mean absolute error.

**4.** Fit a forest on the training rows without `installed`, and print its R² on the test rows next to the full forest's, rounded to three places.

In [5]:
X = network[FEATURES]
y = network["jan_mean"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
forest = RandomForestRegressor(random_state=0).fit(X_train, y_train)

kept = [name for name in FEATURES if name != "installed"]
smaller = RandomForestRegressor(random_state=0).fit(X_train[kept], y_train)

print("all five features:", round(forest.score(X_test, y_test), 3))
print("without installed:", round(smaller.score(X_test[kept], y_test), 3))

all five features: 0.894
without installed: 0.896


The installation year carried no information about January, so the forest without it does as well, 0.896 against 0.894.

**5.** Fit a forest that predicts `snow_on_may_1` from `FEATURES` with `RandomForestClassifier(random_state=0)`, and print the permutation importance of each feature on a held-out quarter of the stations, with `random_state=0` for both the split and the shuffles.

In [6]:
from sklearn.ensemble import RandomForestClassifier

X = network[FEATURES]
snow = network["snow_on_may_1"]
X_train, X_test, snow_train, snow_test = train_test_split(X, snow, test_size=0.25, random_state=0)
classifier = RandomForestClassifier(random_state=0).fit(X_train, snow_train)
result = permutation_importance(classifier, X_test, snow_test, n_repeats=10, random_state=0)

print(pd.Series(result.importances_mean, index=FEATURES).sort_values(ascending=False).round(3))

latitude       0.080
elevation_m    0.066
coast_km       0.061
longitude      0.042
installed     -0.005
dtype: float64


For a classifier, the score is accuracy, so each number is how far the share of test stations the forest classifies correctly falls when the feature is shuffled.

---

&#8592; **Back to:** [Feature Importance](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/11-feature-importance.ipynb)  &nbsp;&middot;&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)
